# Decision-Focused Learning: When and Why SPO+ Helps

This notebook builds understanding layer-by-layer:

**Roadmap:**
1. **The 2-arm problem** — Data distribution and how full vs partial feedback affects MSE and SPO+ predictions
2. **Partial observability breaks things** — Semi-bandit feedback: only selected arms observed. Naive SPO+ fails; masking prevents bad gradients.
3. **Context distribution** — Narrow vs wide context controls flip set lifetime and SPO+ advantage
4. **Masking isn't enough — we need exploration** — Masked SPO+ avoids bad gradients but can't learn about unobserved arms. Bayesian uncertainty enables directed exploration toward contested arms.
5. **The full system** — Shared features + Bayesian + masked SPO+ + UCB, now properly motivated
6. **Component dissection** — Which piece matters most?
7. **Controlling factors** — Margins, context distribution, k/n ratio


In [1]:
import numpy as np
import pandas as pd
from lets_plot import *
from lets_plot.mapping import as_discrete
from scipy.stats import spearmanr
import warnings

LetsPlot.setup_html()
import os
os.makedirs('figures', exist_ok=True)

np.random.seed(1)
LR = 0.001

def solve_topk_min(costs, k):
    sol = np.zeros_like(costs)
    idx = np.argsort(costs)[:k]
    sol[idx] = 1.0
    return sol

def compute_flip_set(pred, true_costs, k):
    solver = lambda c: solve_topk_min(c, k)
    w_true = solver(true_costs)
    w_spo = solver(2 * pred - true_costs)
    flip_mask = np.abs(w_spo - w_true) > 0.5
    return flip_mask, flip_mask.sum()

class BayesianSharedFeatureAgent:
    
    def __init__(self, input_dim, hidden_dim, n_arms, k, lr=None,
                 loss_type='mse', explore='greedy', beta=1.0,
                 lambda_reg=1.0, mask_grad=True, seed=0, decay_rounds=300, beta_end=None,
                 spo_weight=0.5, lr_schedule='constant',
                 phased=False, phase_length=50, phase_epochs=5,
                 mask_full=False, impute_mode='model'):
        lr = lr or LR
        self.n_arms = n_arms
        self.k = k
        self.lr = lr
        self.lr_schedule = lr_schedule
        self.loss_type = loss_type
        self.spo_weight = spo_weight
        self.explore = explore
        self.beta = beta
        self.mask_grad = mask_grad
        self.mask_full = mask_full
        self.impute_mode = impute_mode
        self._cost_sum = np.zeros(n_arms)
        self._cost_count = np.zeros(n_arms)
        self.beta_start = beta
        self.beta_end = beta_end if beta_end is not None else beta
        self.decay_rounds = decay_rounds
        self.t = 0
        self.solver = lambda c: solve_topk_min(c, k)
        self.hidden_dim = hidden_dim
        
        rng = np.random.RandomState(seed)
        self.A = rng.randn(hidden_dim, input_dim) * 0.1
        self.bias = np.zeros(hidden_dim)
        
        self.Lambda = np.array([lambda_reg * np.eye(hidden_dim) for _ in range(n_arms)])
        self.Lambda_inv = np.array([np.eye(hidden_dim) / lambda_reg for _ in range(n_arms)])
        self.b_vec = np.zeros((n_arms, hidden_dim))
        self.lambda_reg = lambda_reg
        self.input_dim = input_dim
        self.phased = phased
        self.phase_length = phase_length
        self.phase_epochs = phase_epochs
        if phased:
            self._buf_x, self._buf_y, self._buf_sel = [], [], []
            self._rounds_since_retrain = 0
    
    def _features(self, x):
        return self.A @ x + self.bias
    
    def _predict(self, h):
        means = np.zeros(self.n_arms)
        for i in range(self.n_arms):
            mu_i = self.Lambda_inv[i] @ self.b_vec[i]
            means[i] = mu_i @ h
        return means
    
    def _uncertainty(self, h):
        stds = np.zeros(self.n_arms)
        for i in range(self.n_arms):
            var = h @ self.Lambda_inv[i] @ h
            stds[i] = np.sqrt(max(var, 0.0))
        return stds
    
    def act(self, x):
        self.t += 1
        if self.beta_start != self.beta_end:
            progress = min(self.t / self.decay_rounds, 1.0)
            self.beta = self.beta_start + (self.beta_end - self.beta_start) * progress
        h = self._features(x)
        means = self._predict(h)
        stds = self._uncertainty(h)
        
        if self.explore == 'greedy':
            return self.solver(means)
        elif self.explore == 'ucb':
            values = means - self.beta * stds
            return self.solver(values)
        return self.solver(means)
    
    def _effective_lr(self):
        if self.lr_schedule == 'inv_sqrt':
            return self.lr / np.sqrt(max(self.t, 1))
        return self.lr
    
    def _compute_grad(self, x, true_costs, obs_mask, full_feedback):
        """Compute feature gradient for one data point."""
        h = self._features(x)
        pred = self._predict(h)
        if self.loss_type == 'mse':
            grad_pred = np.zeros(self.n_arms)
            for i in range(self.n_arms):
                if obs_mask[i]:
                    grad_pred[i] = 2 * (pred[i] - true_costs[i]) / self.n_arms
        elif self.loss_type == 'spo':
            if full_feedback:
                imputed = true_costs.copy()
            elif self.impute_mode == 'mean':
                imputed = np.where(self._cost_count > 0,
                                   self._cost_sum / np.maximum(self._cost_count, 1),
                                   0.5 * np.ones(self.n_arms))
                imputed[obs_mask] = true_costs[obs_mask]
            else:
                imputed = pred.copy()
                imputed[obs_mask] = true_costs[obs_mask]
            w_imp = self.solver(imputed)
            w_spo = self.solver(2 * pred - imputed)
            grad_pred = 2 * (w_imp - w_spo) / self.n_arms
            if self.mask_grad and (not full_feedback or self.mask_full):
                grad_pred = np.where(obs_mask, grad_pred, 0.0)
        grad_h = np.zeros(self.hidden_dim)
        for i in range(self.n_arms):
            mu_i = self.Lambda_inv[i] @ self.b_vec[i]
            grad_h += grad_pred[i] * mu_i
        return grad_h

    def _update_bayesian(self, h, true_costs, obs_mask):
        """Update Bayesian sufficient stats for observed arms."""
        self._cost_sum[obs_mask] += true_costs[obs_mask]
        self._cost_count[obs_mask] += 1
        for i in range(self.n_arms):
            if obs_mask[i]:
                self.Lambda[i] += np.outer(h, h)
                self.b_vec[i] += h * true_costs[i]
                Lih = self.Lambda_inv[i] @ h
                self.Lambda_inv[i] -= np.outer(Lih, Lih) / (1.0 + h @ Lih)

    def _reset_bayesian(self):
        """Reset Bayesian stats (for replay after phased retrain)."""
        self.Lambda = np.array([self.lambda_reg * np.eye(self.hidden_dim) for _ in range(self.n_arms)])
        self.Lambda_inv = np.array([np.eye(self.hidden_dim) / self.lambda_reg for _ in range(self.n_arms)])
        self.b_vec = np.zeros((self.n_arms, self.hidden_dim))

    def _phased_retrain(self):
        """Batch retrain features, then replay buffer to rebuild posteriors."""
        for _epoch in range(self.phase_epochs):
            for x, y, sel in zip(self._buf_x, self._buf_y, self._buf_sel):
                obs_mask = sel > 0.5
                grad_h = self._compute_grad(x, y, obs_mask, False)
                eff_lr = self._effective_lr()
                self.A -= eff_lr * np.outer(grad_h, x)
                self.bias -= eff_lr * grad_h
        # Replay to rebuild posteriors with updated features
        self._reset_bayesian()
        for x, y, sel in zip(self._buf_x, self._buf_y, self._buf_sel):
            h = self._features(x)
            obs_mask = sel > 0.5
            self._update_bayesian(h, y, obs_mask)

    def update(self, x, selection, true_costs, full_feedback=False):
        h = self._features(x)
        if full_feedback:
            obs_mask = np.ones(self.n_arms, dtype=bool)
        else:
            obs_mask = selection > 0.5
        
        self._update_bayesian(h, true_costs, obs_mask)

        if self.phased:
            self._buf_x.append(x.copy())
            self._buf_y.append(true_costs.copy())
            self._buf_sel.append(selection.copy())
            self._rounds_since_retrain += 1
            if self._rounds_since_retrain >= self.phase_length:
                self._phased_retrain()
                self._rounds_since_retrain = 0
        else:
            grad_h = self._compute_grad(x, true_costs, obs_mask, full_feedback)
            eff_lr = self._effective_lr()
            self.A -= eff_lr * np.outer(grad_h, x)
            self.bias -= eff_lr * grad_h



def run_experiment(W_true, k, n_rounds=3000, hidden_dim=8,
                   lr=None, loss_type='mse', explore='greedy',
                   beta=1.0, mask_grad=True, full_feedback=False, seed=0,
                   context_dist='gaussian', ignore_context=False,
                   spo_weight=0.5, lr_schedule='constant',
                 phased=False, phase_length=50, phase_epochs=5,
                   mask_full=False, impute_mode='model'):
    lr = lr or LR
    rng = np.random.RandomState(seed)
    n_arms, input_dim = W_true.shape

    agent = BayesianSharedFeatureAgent(
        input_dim, hidden_dim, n_arms, k, lr=lr,
        loss_type=loss_type, explore=explore, beta=beta,
        mask_grad=mask_grad, seed=seed, spo_weight=spo_weight,
        lr_schedule=lr_schedule, phased=phased,
        phase_length=phase_length, phase_epochs=phase_epochs,
        mask_full=mask_full, impute_mode=impute_mode
    )
    solver = lambda c: solve_topk_min(c, k)

    cum_regret = 0
    regrets, flip_sizes = [], []
    contested_maes, noncontested_maes, decision_correct = [], [], []
    rank_corrs, contested_sigmas = [], []
    obs_counts = np.zeros(n_arms)
    selections_hist, optimals_hist = [], []

    for t in range(n_rounds):
        if context_dist == 'gaussian':
            x = rng.randn(input_dim)
        elif context_dist == 'uniform_narrow':
            x = rng.uniform(-0.5, 0.5, input_dim)
        elif context_dist == 'uniform_wide':
            x = rng.uniform(-3, 3, input_dim)
        else:
            x = rng.randn(input_dim)
        true_costs = 1 / (1 + np.exp(-W_true @ x))
        if ignore_context:
            x = np.zeros(input_dim)  # agent sees no context
        
        selection = agent.act(x)
        optimal = solver(true_costs)
        regret = true_costs @ selection - true_costs @ optimal
        cum_regret += max(regret, 0)
        regrets.append(cum_regret)
        
        obs_counts += (selection > 0.5)
        selections_hist.append((selection > 0.5).astype(int))
        optimals_hist.append((optimal > 0.5).astype(int))
        
        h = agent._features(x)
        pred_costs = agent._predict(h)
        stds = agent._uncertainty(h)
        
        _, n_flips = compute_flip_set(pred_costs, true_costs, k)
        flip_sizes.append(n_flips)
        
        sorted_true = np.sort(true_costs)
        if k < n_arms:
            boundary = (sorted_true[k-1] + sorted_true[k]) / 2
            margin = np.abs(true_costs - boundary)
            n_contested = min(2 * k, n_arms)
            contested_mask = np.zeros(n_arms, dtype=bool)
            contested_mask[np.argsort(margin)[:n_contested]] = True
        else:
            contested_mask = np.ones(n_arms, dtype=bool)
        
        errors = np.abs(pred_costs - true_costs)
        contested_maes.append(errors[contested_mask].mean())
        noncontested_mask = ~contested_mask
        if noncontested_mask.any():
            noncontested_maes.append(errors[noncontested_mask].mean())
        else:
            noncontested_maes.append(0.0)
        
        decision_correct.append(int(np.array_equal(
            (solver(pred_costs) > 0.5).astype(int),
            (optimal > 0.5).astype(int)
        )))
        
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            rho, _ = spearmanr(pred_costs, true_costs)
        rank_corrs.append(rho if np.isfinite(rho) else 0.0)
        contested_sigmas.append(stds[contested_mask].mean())
        
        agent.update(x, selection, true_costs, full_feedback=full_feedback)
    
    return {
        'regrets': np.array(regrets),
        'flip_sizes': np.array(flip_sizes),
        'contested_mae': np.array(contested_maes),
        'noncontested_mae': np.array(noncontested_maes),
        'decision_correct': np.array(decision_correct),
        'rank_corr': np.array(rank_corrs),
        'contested_sigma': np.array(contested_sigmas),
        'obs_counts': obs_counts,
        'selections': np.array(selections_hist),
        'optimals': np.array(optimals_hist),
    }

N_ARMS = 8
CONTEXT_DIM = 5
K = 2  # k/n = 25%
HIDDEN_DIM = 8

rng_gt = np.random.RandomState(42)
W_TRUE = rng_gt.randn(1, CONTEXT_DIM) + rng_gt.randn(N_ARMS, CONTEXT_DIM) * 0.4

def run_context_blind(W_true, k, n_rounds=3000, n_seeds=10, hidden_dim=8, lr=None):
    lr = lr or LR
    results = []
    for seed in range(n_seeds):
        r = run_experiment(W_true, k, n_rounds=n_rounds, hidden_dim=hidden_dim,
                           lr=lr, loss_type='mse', explore='greedy',
                           mask_grad=True, full_feedback=False, seed=seed,
                           ignore_context=True)
        results.append(r)
    return results

In [2]:
N_SEEDS = 10


def train_agent_for_viz(W_true, k, n_rounds=20, hidden_dim=4, loss_type='mse',
                        explore='greedy', mask_grad=True, full_feedback=False,
                        seed=0, beta=1.0, lr=None, lr_schedule='constant',
                        phased=False, phase_length=50, phase_epochs=5,
                        mask_full=False, impute_mode='model'):
    """Train an agent and return it (for visualization of learned predictions)."""
    lr = lr or LR
    rng = np.random.RandomState(seed)
    n_arms, input_dim = W_true.shape

    agent = BayesianSharedFeatureAgent(
        input_dim, hidden_dim, n_arms, k, lr=lr,
        loss_type=loss_type, explore=explore, beta=beta,
        mask_grad=mask_grad, seed=seed, lr_schedule=lr_schedule,
        phased=phased, phase_length=phase_length, phase_epochs=phase_epochs,
        mask_full=mask_full, impute_mode=impute_mode,
    )

    for t in range(n_rounds):
        x = rng.randn(input_dim)
        true_costs = 1 / (1 + np.exp(-W_true @ x))
        selection = agent.act(x)
        agent.update(x, selection, true_costs, full_feedback=full_feedback)

    return agent


def plot_prediction_scatter(agent, W_true, title='', n_samples=300, seed=99,
                            show_context=False):
    """Plot agent predictions."""
    rng = np.random.RandomState(seed)
    n_arms, input_dim = W_true.shape

    if show_context and input_dim == 1:
        x_samples = rng.randn(n_samples, 1)
        x_all, pred_all, arm_labels = [], [], []
        for xi in x_samples:
            h = agent._features(xi.flatten())
            pred_costs = agent._predict(h)
            for i in range(n_arms):
                x_all.append(xi[0])
                pred_all.append(pred_costs[i])
                arm_labels.append(f'Arm {i}')

        df = pd.DataFrame({'x': x_all, 'Predicted': pred_all, 'Arm': arm_labels})
        return (ggplot(df)
            + geom_point(aes('x', 'Predicted', color='Arm'), alpha=0.4, size=1.5)
            + geom_hline(yintercept=0.5, linetype='dashed', alpha=0.3)
            + coord_cartesian(xlim=(-3, 3), ylim=(-0.5, 1))
            + labs(x='Context x', y='Predicted cost', title=title)
            + theme(legend_position='none')
            + ggsize(400, 350)
        )
    else:
        preds_all, trues_all, arms_all = [], [], []
        for _ in range(n_samples):
            x = rng.randn(input_dim)
            true_costs = 1 / (1 + np.exp(-W_true @ x))
            h = agent._features(x)
            pred_costs = agent._predict(h)
            for i in range(n_arms):
                preds_all.append(pred_costs[i])
                trues_all.append(true_costs[i])
                arms_all.append(f'Arm {i}')

        df = pd.DataFrame({'True cost': trues_all, 'Predicted': preds_all, 'Arm': arms_all})
        lims = [min(min(trues_all), min(preds_all)) - 0.05,
                max(max(trues_all), max(preds_all)) + 0.05]
        return (ggplot(df)
            + geom_point(aes('True cost', 'Predicted', color='Arm'), alpha=0.3, size=1)
            + geom_abline(slope=1, intercept=0, linetype='dashed', color='black', alpha=0.5)
            + coord_fixed(xlim=lims, ylim=lims)
            + labs(title=title)
            + theme(legend_position='none')
            + ggsize(400, 350)
        )


def plot_decision_heatmap(result, title='', window=50):
    """Heatmap of per-arm selection frequency over time (rolling window)."""
    selections = result['selections']
    n_rounds, n_arms = selections.shape
    rows = []
    for t in range(window, n_rounds, max(1, n_rounds // 200)):
        freq = selections[t - window:t].mean(axis=0)
        for arm in range(n_arms):
            rows.append({'Round': t, 'Arm': f'Arm {arm}', 'Freq': freq[arm]})
    df = pd.DataFrame(rows)
    return (ggplot(df, aes('Round', 'Arm', fill='Freq'))
        + geom_tile()
        + scale_fill_gradient(low='white', high='#2C3E50', limits=[0, 1])
        + labs(title=title, fill='Select freq')
        + theme(legend_position='right')
        + ggsize(400, 250)
    )

## 1. The 2-Arm Problem: Full vs Partial Feedback

**Setup**: 2 arms, pick 1 (minimize cost). Costs are nonlinear: $c_i(x) = \sigma(w_i^\top x)$.
The agent learns a shared linear feature extractor $h = Ax$ and a Bayesian last layer.

We compare four configurations:
- **MSE (full)**: MSE loss, observes all arms
- **MSE (partial)**: MSE loss, observes only selected arm
- **SPO+ (full)**: SPO+ loss, observes all arms
- **SPO+ (partial)**: SPO+ loss with masking, observes only selected arm

First: the data distribution. Then: how each agent's predictions look after training.

In [3]:
W_TRUE_2ARM = np.array([[1.0], [2.5]])

# --- Data distribution ---
rng_data = np.random.RandomState(42)
n_samples = 300
x_samples = rng_data.randn(n_samples, 1)
costs_samples = 1 / (1 + np.exp(-W_TRUE_2ARM @ x_samples.T))  # (2, n_samples)

df_data = pd.DataFrame({
    'x': np.tile(x_samples.flatten(), 2),
    'cost': np.concatenate([costs_samples[0], costs_samples[1]]),
    'arm': ['Arm 0 (w=1.0)'] * n_samples + ['Arm 1 (w=2.5)'] * n_samples,
})

p_data = (ggplot(df_data)
    + geom_point(aes('x', 'cost', color='arm'), alpha=0.4, size=1.5)
    + scale_color_manual(values={'Arm 0 (w=1.0)': '#3498DB', 'Arm 1 (w=2.5)': '#E74C3C'})
    + geom_hline(yintercept=0.5, linetype='dashed', alpha=0.3)
    + coord_cartesian(xlim=(-3, 3), ylim=(-0.5, 1))
    + labs(x='Context x', y='True cost c(x)', title='Data Distribution: 2 Arms', color='')
    + theme(legend_position='top')
    + ggsize(400, 350)
)

# --- Configs ---
n_rounds_2arm = 5000
configs_2arm = [
    # (name, kwargs)
    ('MSE (full)',              dict(loss_type='mse', full_feedback=True)),
    ('MSE (partial)',           dict(loss_type='mse', full_feedback=False)),
    ('SPO+ (full)',             dict(loss_type='spo', full_feedback=True)),
    ('SPO+ (full, force mask)', dict(loss_type='spo', full_feedback=True, mask_full=True)),
    ('SPO+ (partial)',          dict(loss_type='spo', full_feedback=False)),
    ('SPO+ (partial, mean imp)', dict(loss_type='spo', full_feedback=False, impute_mode='mean')),
    ('SPO+-Phased (partial)',   dict(loss_type='spo', full_feedback=False, phased=True)),
]

scatter_plots_2arm = [p_data]
regret_rows = []

for name, kw in configs_2arm:
    agent = train_agent_for_viz(W_TRUE_2ARM, 1, n_rounds=n_rounds_2arm,
                                hidden_dim=4, explore='ucb', mask_grad=True,
                                seed=0, **kw)
    scatter_plots_2arm.append(
        plot_prediction_scatter(agent, W_TRUE_2ARM, title=name, n_samples=300,
                                seed=99, show_context=True)
    )

    for seed in range(N_SEEDS):
        r = run_experiment(W_TRUE_2ARM, 1, n_rounds=n_rounds_2arm, hidden_dim=4,
                           lr=LR, explore='ucb', mask_grad=True, seed=seed, **kw)
        cum_regret = r['regrets']
        inst_regret = np.diff(cum_regret, prepend=0)
        for t in range(n_rounds_2arm):
            regret_rows.append({
                'Round': t, 'Cumulative Regret': cum_regret[t],
                'Instant Regret': inst_regret[t],
                'Agent': name, 'Seed': seed,
            })

df_regret = pd.DataFrame(regret_rows)
agg = df_regret.groupby(['Round', 'Agent']).agg(
    cum_mean=('Cumulative Regret', 'mean'),
    inst_mean=('Instant Regret', 'mean'),
).reset_index()

# Row 1: prediction scatter
row1 = gggrid(scatter_plots_2arm[:5], ncol=5) + ggsize(1800, 380)
row1b = gggrid(scatter_plots_2arm[5:], ncol=4) + ggsize(1800, 380)
display(row1)
display(row1b)

# Row 2: regret curves
p_cum = (ggplot(agg, aes('Round', 'cum_mean', color='Agent'))
    + geom_line(size=1.2)
    + labs(x='Round', y='Cumulative Regret', title='Cumulative Regret')
    + theme(legend_position='none')
    + ggsize(800, 350)
)

p_inst = (ggplot(agg, aes('Round', 'inst_mean', color='Agent'))
    + geom_line(size=1.2, alpha=0.8)
    + labs(x='Round', y='Per-Round Regret', title='Per-Round Regret')
    + theme(legend_position='bottom')
    + ggsize(800, 350)
)

row2 = gggrid([p_cum, p_inst], ncol=2) + ggsize(1800, 380)
display(row2)

ggsave(row1, 'figures/sec1_2arm_predictions_row1.svg')
ggsave(row1b, 'figures/sec1_2arm_predictions_row2.svg')
ggsave(row2, 'figures/sec1_2arm_regret.svg')

# Print final regret summary
for name, kw in configs_2arm:
    final = df_regret[(df_regret['Agent'] == name) & (df_regret['Round'] == n_rounds_2arm - 1)]
    regs = final['Cumulative Regret'].values
    print(f"{name:30s}: regret = {np.mean(regs):.1f} +/- {np.std(regs):.1f}")

MSE (full)                    : regret = 310.1 +/- 5.2
MSE (partial)                 : regret = 326.2 +/- 3.0
SPO+ (full)                   : regret = 0.1 +/- 0.0
SPO+ (full, force mask)       : regret = 0.1 +/- 0.0
SPO+ (partial)                : regret = 335.4 +/- 6.2
SPO+ (partial, mean imp)      : regret = 286.5 +/- 2.5
SPO+-Phased (partial)         : regret = 114.1 +/- 8.9


## 2. Partial Observability Breaks Naive SPO+

Now remove full information. Under **semi-bandit** feedback, we only observe
costs for the k=2 arms we selected (out of 8). For the other 6, we must
**impute** their costs using our current predictions.

This creates a fundamental problem for SPO+:
- SPO+ computes $w^*(2\hat{c} - c)$ where $c$ includes imputed values for unobserved arms
- If imputation is wrong, the SPO+ perturbation points in the wrong direction
- This can cause **catastrophic failure**: the agent locks onto a bad decision

**Four agents compared** (same 8-arm architecture, only feedback/masking changes):
- **SPO+ (full info)**: Oracle — observes all arms (upper bound)
- **MSE (semi-bandit)**: Only updates observed arms — naturally handles partial feedback
- **SPO+ (no mask)**: Imputes with predictions, applies gradient to ALL arms — catastrophic failure expected
- **SPO+ (masked)**: Imputes with predictions, but **zeros gradient on unobserved arms** — safe but limited

Plots show regret (with context-blind baseline), zoomed regret, and flip set dynamics.


In [4]:
n_rounds_sb = 3000

sb_configs = [
    # (name, loss, full_feedback, mask_grad, _, phased)
    ('SPO+ (full info)',     'spo', True,  True,  True,  False),
    ('MSE (semi-bandit)',    'mse', False, True,  False, False),
    ('SPO+ (no mask)',       'spo', False, False, False, False),
    ('SPO+ (masked)',        'spo', False, True,  False, False),
    ('SPO+ (masked, phased)','spo', False, True,  False, True),
    ('MSE (phased)',         'mse', False, True,  False, True),
]

results_sb = {}
for name, loss, ff, mask, _, phased in sb_configs:
    print(f'Running {name}...')
    seeds_results = []
    for seed in range(N_SEEDS):
        r = run_experiment(W_TRUE, K, n_rounds=n_rounds_sb, hidden_dim=HIDDEN_DIM,
                           lr=LR, loss_type=loss, explore='ucb',
                           mask_grad=mask, full_feedback=ff, seed=seed,
                           phased=phased)
        seeds_results.append(r)
    results_sb[name] = seeds_results

cb_results_8 = run_context_blind(W_TRUE, K, n_rounds=n_rounds_sb, n_seeds=N_SEEDS, hidden_dim=HIDDEN_DIM, lr=LR)

sb_colors = {
    'SPO+ (full info)': '#1A5276',
    'MSE (semi-bandit)': '#C0392B',
    'SPO+ (no mask)': '#2E86C1',
    'SPO+ (masked)': '#5DADE2',
    'SPO+ (masked, phased)': '#27AE60',
    'MSE (phased)': '#F39C12',
}

# (a) Regret with context-blind
regret_dfs = []
for name, _, _, _, _, _ in sb_configs:
    regs = np.array([r['regrets'] for r in results_sb[name]])
    mean_reg = regs.mean(axis=0)
    std_reg = regs.std(axis=0)
    regret_dfs.append(pd.DataFrame({
        'round': np.arange(n_rounds_sb), 'regret': mean_reg,
        'lower': mean_reg - std_reg, 'upper': mean_reg + std_reg, 'agent': name,
    }))
cb_regs_8 = np.array([r['regrets'] for r in cb_results_8])
regret_dfs.append(pd.DataFrame({
    'round': np.arange(n_rounds_sb), 'regret': cb_regs_8.mean(axis=0),
    'lower': cb_regs_8.mean(axis=0) - cb_regs_8.std(axis=0),
    'upper': cb_regs_8.mean(axis=0) + cb_regs_8.std(axis=0),
    'agent': 'Context-Blind',
}))
df_reg = pd.concat(regret_dfs, ignore_index=True)
all_colors = {**sb_colors, 'Context-Blind': 'gray'}

p_a = (ggplot(df_reg)
    + geom_line(aes('round', 'regret', color='agent'), size=1.2)
    + geom_ribbon(aes('round', ymin='lower', ymax='upper', fill='agent'), alpha=0.15, color='rgba(0,0,0,0)')
    + scale_color_manual(values=all_colors) + scale_fill_manual(values=all_colors)
    + labs(x='Round', y='Cumulative Regret', title=f'(a) With Context-Blind Baseline (n={N_ARMS}, k={K})')
    + theme(legend_position='bottom')
    + ggsize(450, 350)
)

# (b) Zoomed
df_zoom = df_reg[df_reg['agent'] != 'Context-Blind']
p_b = (ggplot(df_zoom)
    + geom_line(aes('round', 'regret', color='agent'), size=1.2)
    + geom_ribbon(aes('round', ymin='lower', ymax='upper', fill='agent'), alpha=0.15, color='rgba(0,0,0,0)')
    + scale_color_manual(values=sb_colors) + scale_fill_manual(values=sb_colors)
    + labs(x='Round', y='Cumulative Regret', title='(b) Zoomed (no baseline)')
    + theme(legend_position='bottom')
    + ggsize(450, 350)
)

# (c) Per-round regret
perround_dfs = []
for name, _, _, _, _, _ in sb_configs:
    regs = np.array([r['regrets'] for r in results_sb[name]])
    per_round = np.diff(regs, axis=1, prepend=0)
    smoothed = np.convolve(per_round.mean(axis=0), np.ones(50)/50, mode='valid')
    perround_dfs.append(pd.DataFrame({
        'round': np.arange(len(smoothed)), 'regret': smoothed, 'agent': name,
    }))
cb_per = np.diff(cb_regs_8, axis=1, prepend=0)
cb_sm = np.convolve(cb_per.mean(axis=0), np.ones(50)/50, mode='valid')
perround_dfs.append(pd.DataFrame({
    'round': np.arange(len(cb_sm)), 'regret': cb_sm, 'agent': 'Context-Blind',
}))
df_pr = pd.concat(perround_dfs, ignore_index=True)

p_c = (ggplot(df_pr)
    + geom_line(aes('round', 'regret', color='agent'), size=1.2)
    + scale_color_manual(values=all_colors)
    + labs(x='Round', y='Per-Round Regret (rolling avg)', title='(c) Per-Round Regret')
    + theme(legend_position='bottom')
    + ggsize(450, 350)
)

# (d) Flip set
window_sb = 50
kernel_sb = np.ones(window_sb) / window_sb
flip_dfs = []
for name, _, _, _, _, _ in sb_configs:
    flips = np.array([r['flip_sizes'] for r in results_sb[name]])
    smoothed = np.convolve(flips.mean(axis=0), kernel_sb, mode='valid')
    flip_dfs.append(pd.DataFrame({
        'round': np.arange(len(smoothed)), 'flip_size': smoothed, 'agent': name,
    }))
df_flip = pd.concat(flip_dfs, ignore_index=True)

p_d = (ggplot(df_flip)
    + geom_line(aes('round', 'flip_size', color='agent'), size=1.2)
    + scale_color_manual(values=sb_colors)
    + labs(x='Round', y='Flip Set Size (rolling avg)', title='(d) Flip Set Under Partial Observability')
    + theme(legend_position='bottom')
    + ggsize(450, 350)
)

display(gggrid([p_a, p_b, p_c, p_d], ncol=4) + ggsize(1800, 400))
ggsave(gggrid([p_a, p_b, p_c, p_d], ncol=4) + ggsize(1800, 400), 'figures/sec5_regret_flip_accuracy.svg')

for name, _, _, _, _, _ in sb_configs:
    final = [r['regrets'][-1] for r in results_sb[name]]
    print(f"{name:25s}: regret = {np.mean(final):.1f} +/- {np.std(final):.1f}")


Running SPO+ (full info)...
Running MSE (semi-bandit)...
Running SPO+ (no mask)...
Running SPO+ (masked)...
Running SPO+ (masked, phased)...
Running MSE (phased)...


SPO+ (full info)         : regret = 7.2 +/- 1.3
MSE (semi-bandit)        : regret = 321.6 +/- 72.3
SPO+ (no mask)           : regret = 692.0 +/- 6.5
SPO+ (masked)            : regret = 180.6 +/- 30.4
SPO+ (masked, phased)    : regret = 59.9 +/- 3.5
MSE (phased)             : regret = 74.8 +/- 7.0


In [5]:
sb_viz_configs = [
    # (name, loss, full_feedback, mask_grad, phased)
    ('SPO+ (full info)',     'spo', True,  True,  False),
    ('MSE (semi-bandit)',    'mse', False, True,  False),
    ('SPO+ (no mask)',       'spo', False, False, False),
    ('SPO+ (masked)',        'spo', False, True,  False),
    ('SPO+ (masked, phased)','spo', False, True,  True),
    ('MSE (phased)',         'mse', False, True,  True),
]

scatter_plots = []
heatmap_plots = []
for name, loss, ff, mask, phased in sb_viz_configs:
    agent = train_agent_for_viz(W_TRUE, K, n_rounds=n_rounds_sb,
                                hidden_dim=HIDDEN_DIM, loss_type=loss,
                                explore='ucb', mask_grad=mask,
                                full_feedback=ff, seed=0, phased=phased)
    scatter_plots.append(plot_prediction_scatter(agent, W_TRUE, title=name))
    r = results_sb[name][0]
    heatmap_plots.append(plot_decision_heatmap(r, title=name))

display(gggrid(scatter_plots, ncol=3) + ggsize(1050, 760))
ggsave(gggrid(scatter_plots, ncol=3) + ggsize(1050, 760), 'figures/sec5_prediction_scatter.svg')
display(gggrid(heatmap_plots, ncol=3) + ggsize(1050, 500))
ggsave(gggrid(heatmap_plots, ncol=3) + ggsize(1050, 500), 'figures/sec5_decision_heatmap.svg')

'/Users/aabhash/Documents/code/DFL for TS Autoregressor/constrained-bandits/notebooks/lets-plot-images/figures/sec5_decision_heatmap.svg'

## 3. Context Distribution Controls Flip Set Lifetime

The context distribution determines how often the agent sees near-boundary contexts:
- **Narrow U[-0.5, 0.5]**: Most contexts are near the decision boundary — flip set stays alive longer, giving SPO+ more rounds to exploit decision-focused gradients
- **Gaussian N(0, 1)**: Standard spread — moderate near-boundary frequency
- **Wide U[-3, 3]**: Most contexts are far from the boundary — decisions are easy, flip set collapses quickly, SPO+ advantage vanishes

This directly controls how long SPO+ maintains its advantage over MSE. Runs MSE (semi-bandit), SPO+ without masking, and SPO+ with masking across all three distributions. Top row: cumulative regret. Bottom row: smoothed flip set size over time.


In [6]:
n_rounds_ctx = 5000

context_dists = ['uniform_narrow', 'gaussian', 'uniform_wide']
dist_labels = ['Narrow U[-0.5, 0.5]', 'Gaussian N(0, 1)', 'Wide U[-3, 3]']

ctx_agents = [
    # (name, loss, explore, beta, mask_grad, phased)
    ('MSE (semi-bandit)',    'mse', 'ucb', 1.0, True,  False),
    ('SPO+ (no mask)',       'spo', 'ucb', 1.0, False, False),
    ('SPO+ (masked)',        'spo', 'ucb', 1.0, True,  False),
    ('SPO+ (masked, phased)','spo', 'ucb', 1.0, True,  True),
    ('MSE (phased)',         'mse', 'ucb', 1.0, True,  True),
]
ctx_colors = {
    'MSE (semi-bandit)': '#C0392B',
    'SPO+ (no mask)': '#2E86C1',
    'SPO+ (masked)': '#1A5276',
    'SPO+ (masked, phased)': '#27AE60',
    'MSE (phased)': '#F39C12',
}

n_ctx_seeds = 5
ctx_results = {}

for cdist, dlabel in zip(context_dists, dist_labels):
    print(f'\n--- {dlabel} ---')
    for name, loss, explore, beta, mask, phased in ctx_agents:
        print(f'  {name}...', end=' ', flush=True)
        seeds_res = []
        for seed in range(n_ctx_seeds):
            r = run_experiment(W_TRUE, K, n_rounds=n_rounds_ctx, hidden_dim=HIDDEN_DIM,
                               lr=LR, loss_type=loss, explore=explore, beta=beta,
                               mask_grad=mask, full_feedback=False, seed=seed,
                               context_dist=cdist, phased=phased)
            seeds_res.append(r)
        ctx_results[(name, cdist)] = seeds_res
        final_reg = np.mean([r['regrets'][-1] for r in seeds_res])
        print(f'regret={final_reg:.0f}')

window_ctx = 50
kernel_ctx = np.ones(window_ctx) / window_ctx

regret_plots = []
perround_plots = []
flip_plots = []

for cdist, dlabel in zip(context_dists, dist_labels):
    # Regret
    reg_dfs = []
    for name, _, _, _, _, _ in ctx_agents:
        regs = np.array([r['regrets'] for r in ctx_results[(name, cdist)]])
        mean_reg = regs.mean(axis=0)
        std_reg = regs.std(axis=0)
        reg_dfs.append(pd.DataFrame({
            'round': np.arange(n_rounds_ctx), 'regret': mean_reg,
            'lower': mean_reg - std_reg, 'upper': mean_reg + std_reg, 'agent': name,
        }))
    df_r = pd.concat(reg_dfs, ignore_index=True)
    p_r = (ggplot(df_r)
        + geom_line(aes('round', 'regret', color='agent'), size=1.2)
        + geom_ribbon(aes('round', ymin='lower', ymax='upper', fill='agent'), alpha=0.15, color='rgba(0,0,0,0)')
        + scale_color_manual(values=ctx_colors) + scale_fill_manual(values=ctx_colors)
        + labs(x='Round', y='Cumulative Regret', title=f'Regret -- {dlabel}')
        + theme(legend_position='bottom')
        + ggsize(400, 300)
    )
    regret_plots.append(p_r)

    # Per-round regret
    pr_dfs = []
    for name, _, _, _, _, _ in ctx_agents:
        regs = np.array([r['regrets'] for r in ctx_results[(name, cdist)]])
        per_round = np.diff(regs, axis=1, prepend=0)
        smoothed = np.convolve(per_round.mean(axis=0), kernel_ctx, mode='valid')
        pr_dfs.append(pd.DataFrame({
            'round': np.arange(len(smoothed)), 'regret': smoothed, 'agent': name,
        }))
    df_pr = pd.concat(pr_dfs, ignore_index=True)
    p_pr = (ggplot(df_pr)
        + geom_line(aes('round', 'regret', color='agent'), size=1.2)
        + scale_color_manual(values=ctx_colors)
        + labs(x='Round', y='Per-Round Regret', title=f'Per-Round -- {dlabel}')
        + theme(legend_position='bottom')
        + ggsize(400, 300)
    )
    perround_plots.append(p_pr)

    # Flip set
    fl_dfs = []
    for name, _, _, _, _, _ in ctx_agents:
        flips = np.array([r['flip_sizes'] for r in ctx_results[(name, cdist)]])
        smoothed = np.convolve(flips.mean(axis=0), kernel_ctx, mode='valid')
        fl_dfs.append(pd.DataFrame({
            'round': np.arange(len(smoothed)), 'flip_size': smoothed, 'agent': name,
        }))
    df_f = pd.concat(fl_dfs, ignore_index=True)
    p_f = (ggplot(df_f)
        + geom_line(aes('round', 'flip_size', color='agent'), size=1.2)
        + scale_color_manual(values=ctx_colors)
        + labs(x='Round', y='Flip Set Size', title=f'Flip Set -- {dlabel}')
        + theme(legend_position='bottom')
        + ggsize(400, 300)
    )
    flip_plots.append(p_f)

display(gggrid(regret_plots + perround_plots + flip_plots, ncol=3) + ggsize(1200, 950))
ggsave(gggrid(regret_plots + perround_plots + flip_plots, ncol=3) + ggsize(1200, 950), 'figures/sec5_context_distribution.svg')

print(f"\n{'Distribution':<22s} {'Agent':<20s} {'Final Regret':>13s} {'Flip End':>10s}")
print('-' * 70)
for cdist, dlabel in zip(context_dists, dist_labels):
    for name, _, _, _, _, _ in ctx_agents:
        reg = np.mean([r['regrets'][-1] for r in ctx_results[(name, cdist)]])
        flips_end = np.mean([r['flip_sizes'][-200:].mean() for r in ctx_results[(name, cdist)]])
        print(f'{dlabel:<22s} {name:<20s} {reg:13.1f} {flips_end:10.2f}')

print('\nNarrow context: near-boundary contexts dominate -> flip set persists -> SPO+ advantage largest.')
print('Wide context: far-boundary contexts dominate -> flip set collapses fast -> SPO+ advantage smallest.')



--- Narrow U[-0.5, 0.5] ---
  MSE (semi-bandit)... regret=301
  SPO+ (no mask)... regret=559
  SPO+ (masked)... regret=173
  SPO+ (masked, phased)... regret=77
  MSE (phased)... regret=99

--- Gaussian N(0, 1) ---
  MSE (semi-bandit)... regret=392
  SPO+ (no mask)... regret=1154
  SPO+ (masked)... regret=225
  SPO+ (masked, phased)... regret=73
  MSE (phased)... regret=92

--- Wide U[-3, 3] ---
  MSE (semi-bandit)... regret=579
  SPO+ (no mask)... regret=1345
  SPO+ (masked)... regret=323
  SPO+ (masked, phased)... regret=96
  MSE (phased)... regret=112



Distribution           Agent                 Final Regret   Flip End
----------------------------------------------------------------------
Narrow U[-0.5, 0.5]    MSE (semi-bandit)            300.8       3.38
Narrow U[-0.5, 0.5]    SPO+ (no mask)               558.9       4.00
Narrow U[-0.5, 0.5]    SPO+ (masked)                173.1       3.19
Narrow U[-0.5, 0.5]    SPO+ (masked, phased)          77.4       0.19
Narrow U[-0.5, 0.5]    MSE (phased)                  99.1       1.34
Gaussian N(0, 1)       MSE (semi-bandit)            391.5       3.73
Gaussian N(0, 1)       SPO+ (no mask)              1153.9       4.00
Gaussian N(0, 1)       SPO+ (masked)                224.6       3.14
Gaussian N(0, 1)       SPO+ (masked, phased)          72.6       1.74
Gaussian N(0, 1)       MSE (phased)                  92.5       0.97
Wide U[-3, 3]          MSE (semi-bandit)            579.0       3.65
Wide U[-3, 3]          SPO+ (no mask)              1344.6       4.00
Wide U[-3, 3]          SPO+ (

## 4. Masking Isn't Enough — We Need Exploration

Masking prevents bad gradients, but creates a new problem: **SPO+ can only learn from observed arms**.

With greedy selection, the agent always picks the same arms (the ones it currently thinks are cheapest).
The **contested arms** — near the selection boundary, where SPO+ gradients are most
informative — may never be observed.

**This is why we need the Bayesian layer:**
- A Bayesian posterior $p(v_i | D_t)$ gives per-arm uncertainty $\sigma_i$
- Unobserved arms have high uncertainty — UCB selects them for exploration
- This ensures contested arms are eventually observed
- Once observed, masked SPO+ can compute informative gradients

The Bayesian layer bridges partial observability: **exploration generates the observations
that masking needs to compute decision-focused gradients.**

Same 8-arm problem, same architecture. 2x2 comparison: {MSE, SPO+} x {Greedy, UCB}. Eight diagnostic panels: regret (with/without context-blind baseline), flip set size, observation coverage, decision accuracy, contested/non-contested MAE, and rank correlation.


In [7]:
n_rounds_explore = 3000

explore_configs = [
    # (name, loss, explore, beta, mask_grad, phased)
    ('MSE + Greedy',           'mse', 'greedy', 0.0, True, False),
    ('MSE + UCB',              'mse', 'ucb',    1.0, True, False),
    ('SPO+ masked + Greedy',   'spo', 'greedy', 0.0, True, False),
    ('SPO+ masked + UCB',      'spo', 'ucb',    1.0, True, False),
    ('SPO+ masked + UCB (phased)', 'spo', 'ucb', 1.0, True, True),
    ('MSE + UCB (phased)',     'mse', 'ucb',    1.0, True, True),
]

explore_results = {}
for name, loss, explore, beta, mask, phased in explore_configs:
    print(f'Running {name}...')
    seeds = []
    for seed in range(N_SEEDS):
        r = run_experiment(W_TRUE, K, n_rounds=n_rounds_explore, hidden_dim=HIDDEN_DIM,
                           lr=LR, loss_type=loss, explore=explore, beta=beta,
                           mask_grad=mask, full_feedback=False, seed=seed,
                           phased=phased)
        seeds.append(r)
    explore_results[name] = seeds

cb_explore = run_context_blind(W_TRUE, K, n_rounds=n_rounds_explore, n_seeds=N_SEEDS, hidden_dim=HIDDEN_DIM, lr=LR)
print('Done')

ex_colors = {
    'MSE + Greedy': '#C0392B',
    'MSE + UCB': '#F1948A',
    'SPO+ masked + Greedy': '#1A5276',
    'SPO+ masked + UCB': '#5DADE2',
    'SPO+ masked + UCB (phased)': '#27AE60',
    'MSE + UCB (phased)': '#F39C12',
    'Context-Blind': 'gray',
}
ex_linestyles = {
    'MSE + Greedy': 'solid', 'MSE + UCB': 'dashed',
    'SPO+ masked + Greedy': 'solid', 'SPO+ masked + UCB': 'dashed',
    'SPO+ masked + UCB (phased)': 'solid',
    'MSE + UCB (phased)': 'solid',
    'Context-Blind': 'dotted',
}

window_ex = 100
kernel_ex = np.ones(window_ex) / window_ex

def make_line_plot(metric_key, ylabel, title, include_cb=True, smooth=True):
    dfs = []
    for name, _, _, _, _, _ in explore_configs:
        vals = np.array([r[metric_key] for r in explore_results[name]])
        mean_v = vals.mean(axis=0)
        if smooth:
            mean_v = np.convolve(mean_v, kernel_ex, mode='valid')
        std_v = vals.std(axis=0)
        if smooth:
            std_v = np.convolve(std_v, kernel_ex, mode='valid')
        dfs.append(pd.DataFrame({
            'round': np.arange(len(mean_v)), 'value': mean_v,
            'lower': mean_v - std_v, 'upper': mean_v + std_v, 'agent': name,
        }))
    if include_cb:
        cb_vals = np.array([r[metric_key] for r in cb_explore])
        cb_mean = cb_vals.mean(axis=0)
        if smooth:
            cb_mean = np.convolve(cb_mean, kernel_ex, mode='valid')
        dfs.append(pd.DataFrame({
            'round': np.arange(len(cb_mean)), 'value': cb_mean,
            'lower': cb_mean, 'upper': cb_mean, 'agent': 'Context-Blind',
        }))
    df = pd.concat(dfs, ignore_index=True)
    return (ggplot(df)
        + geom_line(aes('round', 'value', color='agent', linetype='agent'), size=1.0)
        + scale_color_manual(values=ex_colors)
        + scale_linetype_manual(values=ex_linestyles)
        + labs(x='Round', y=ylabel, title=title)
        + theme(legend_position='bottom', legend_title=element_blank())
        + ggsize(380, 280)
    )

# Regret plots (not smoothed)
def make_regret_plot(title, include_cb=True):
    dfs = []
    for name, _, _, _, _, _ in explore_configs:
        regs = np.array([r['regrets'] for r in explore_results[name]])
        mean_v = regs.mean(axis=0)
        std_v = regs.std(axis=0)
        dfs.append(pd.DataFrame({
            'round': np.arange(n_rounds_explore), 'value': mean_v,
            'lower': mean_v - std_v, 'upper': mean_v + std_v, 'agent': name,
        }))
    if include_cb:
        cb_regs = np.array([r['regrets'] for r in cb_results_8])
        dfs.append(pd.DataFrame({
            'round': np.arange(len(cb_regs[0])), 'value': cb_regs.mean(axis=0),
            'lower': cb_regs.mean(axis=0) - cb_regs.std(axis=0),
            'upper': cb_regs.mean(axis=0) + cb_regs.std(axis=0),
            'agent': 'Context-Blind',
        }))
    df = pd.concat(dfs, ignore_index=True)
    return (ggplot(df)
        + geom_line(aes('round', 'value', color='agent', linetype='agent'), size=1.0)
        + geom_ribbon(aes('round', ymin='lower', ymax='upper', fill='agent'), alpha=0.1, color='rgba(0,0,0,0)')
        + scale_color_manual(values=ex_colors) + scale_fill_manual(values=ex_colors)
        + scale_linetype_manual(values=ex_linestyles)
        + labs(x='Round', y='Cumulative Regret', title=title)
        + theme(legend_position='bottom', legend_title=element_blank())
        + ggsize(380, 280)
    )

def make_perround_regret_plot(title, include_cb=True):
    dfs = []
    for name, _, _, _, _, _ in explore_configs:
        regs = np.array([r['regrets'] for r in explore_results[name]])
        per_round = np.diff(regs, axis=1, prepend=0)
        sv = np.convolve(per_round.mean(axis=0), kernel_ex, mode='valid')
        dfs.append(pd.DataFrame({
            'round': np.arange(len(sv)), 'value': sv, 'agent': name,
        }))
    if include_cb:
        cb_regs = np.array([r['regrets'] for r in cb_explore])
        cb_per = np.diff(cb_regs, axis=1, prepend=0)
        cb_sv = np.convolve(cb_per.mean(axis=0), kernel_ex, mode='valid')
        dfs.append(pd.DataFrame({
            'round': np.arange(len(cb_sv)), 'value': cb_sv, 'agent': 'Context-Blind',
        }))
    df = pd.concat(dfs, ignore_index=True)
    return (ggplot(df)
        + geom_line(aes('round', 'value', color='agent', linetype='agent'), size=1.0)
        + scale_color_manual(values=ex_colors)
        + scale_linetype_manual(values=ex_linestyles)
        + labs(x='Round', y='Per-Round Regret', title=title)
        + theme(legend_position='bottom', legend_title=element_blank())
        + ggsize(380, 280)
    )

p_a = make_regret_plot(f'(a) With Context-Blind Baseline (n={N_ARMS}, k={K})', include_cb=True)
p_b = make_regret_plot('(b) Zoomed (no baseline)', include_cb=False)
p_c = make_perround_regret_plot('(c) Per-Round Regret', include_cb=True)
p_d = make_line_plot('flip_sizes', 'Flip Set Size (rolling avg)', '(d) Flip Set Size |F_t|')

# (d) Observation Coverage bar chart
obs_threshold = n_rounds_explore * K / (2 * N_ARMS)
bar_data = []
for name, _, _, _, _, _ in explore_configs:
    obs = np.array([r['obs_counts'] for r in explore_results[name]])
    coverage = (obs > obs_threshold).mean(axis=1)
    bar_data.append({'agent': name.replace(' + ', '\n'), 'coverage': coverage.mean(), 'std': coverage.std()})
cb_obs = np.array([r['obs_counts'] for r in cb_explore])
cb_cov = (cb_obs > obs_threshold).mean(axis=1)
bar_data.append({'agent': 'Context-\nBlind', 'coverage': cb_cov.mean(), 'std': cb_cov.std()})
df_bar = pd.DataFrame(bar_data)
df_bar['ymin'] = df_bar['coverage'] - df_bar['std']
df_bar['ymax'] = df_bar['coverage'] + df_bar['std']

p_e = (ggplot(df_bar)
    + geom_bar(aes('agent', 'coverage', fill='agent'), stat='identity', alpha=0.8)
    + geom_errorbar(aes('agent', ymin='ymin', ymax='ymax'), width=0.3)
    + scale_fill_manual(values={
        'MSE\nGreedy': '#C0392B', 'MSE\nUCB': '#F1948A',
        'SPO+ masked\nGreedy': '#1A5276', 'SPO+ masked\nUCB': '#5DADE2',
        'Context-\nBlind': 'gray'})
    + ylim(0, 1.1)
    + labs(y=f'Fraction of arms observed >{int(obs_threshold)}x', title='(e) Observation Coverage')
    + theme(legend_position='none', axis_title_x=element_blank())
    + ggsize(380, 280)
)

p_f = make_line_plot('decision_correct', 'Decision Accuracy (rolling avg)', '(f) Decision Accuracy')
p_g = make_line_plot('contested_mae', 'MAE (rolling avg)', '(g) Contested-Arm MAE')
p_h = make_line_plot('noncontested_mae', 'MAE (rolling avg)', '(h) Non-Contested-Arm MAE')
p_i = make_line_plot('rank_corr', 'Spearman rho (rolling avg)', '(i) Rank Correlation')
p_j = make_line_plot('contested_sigma', 'Mean sigma (rolling avg)', '(j) Contested-Arm Posterior sigma')

display(gggrid([p_a, p_b, p_c, p_d, p_e, p_f, p_g, p_h, p_i, p_j], ncol=3) + ggsize(1200, 1100))
ggsave(gggrid([p_a, p_b, p_c, p_d, p_e, p_f, p_g, p_h, p_i, p_j], ncol=3) + ggsize(1200, 1100), 'figures/sec5_exploration.svg')

print(f"\n{'Agent':30s} {'Regret':>10s} {'Coverage':>8s} {'DecAcc (last 500)':>18s}")
print('-' * 70)
for name, _, _, _, _, _ in explore_configs:
    reg = np.mean([r['regrets'][-1] for r in explore_results[name]])
    coverage = np.mean([(r['obs_counts'] > obs_threshold).mean() for r in explore_results[name]])
    dacc = np.mean([r['decision_correct'][-500:].mean() for r in explore_results[name]])
    print(f'{name:30s} {reg:10.1f} {coverage:8.2f} {dacc:18.3f}')

print(f'\nNote: with changing context, greedy observes many arms naturally (different contexts')
print(f'make different arms cheapest). But the WORST arm is much less observed under greedy.')
print(f'UCB ensures even the worst-covered arm gets adequate observations.')


Running MSE + Greedy...
Running MSE + UCB...
Running SPO+ masked + Greedy...
Running SPO+ masked + UCB...
Running SPO+ masked + UCB (phased)...
Running MSE + UCB (phased)...
Done



Agent                              Regret Coverage  DecAcc (last 500)
----------------------------------------------------------------------
MSE + Greedy                        303.1     0.86              0.467
MSE + UCB                           321.6     0.99              0.448
SPO+ masked + Greedy                186.2     0.79              0.546
SPO+ masked + UCB                   180.6     0.94              0.593
SPO+ masked + UCB (phased)           59.9     1.00              0.780
MSE + UCB (phased)                   74.8     0.97              0.692

Note: with changing context, greedy observes many arms naturally (different contexts
make different arms cheapest). But the WORST arm is much less observed under greedy.
UCB ensures even the worst-covered arm gets adequate observations.


In [8]:
# Prediction scatter & decision heatmaps for Section 4
ex_viz_configs = [
    # (name, loss, explore, beta, mask_grad, phased)
    ('MSE + Greedy',             'mse', 'greedy', 0.0, True, False),
    ('MSE + UCB',                'mse', 'ucb',    1.0, True, False),
    ('SPO+ masked + Greedy',     'spo', 'greedy', 0.0, True, False),
    ('SPO+ masked + UCB',        'spo', 'ucb',    1.0, True, False),
    ('SPO+ masked + UCB (phased)','spo', 'ucb',   1.0, True, True),
    ('MSE + UCB (phased)',       'mse', 'ucb',    1.0, True, True),
]

scatter_plots = []
heatmap_plots = []
for name, loss, explore, beta, mask, phased in ex_viz_configs:
    agent = train_agent_for_viz(W_TRUE, K, n_rounds=n_rounds_explore,
                                hidden_dim=HIDDEN_DIM, loss_type=loss,
                                explore=explore, beta=beta, mask_grad=mask,
                                full_feedback=False, seed=0, phased=phased)
    scatter_plots.append(plot_prediction_scatter(agent, W_TRUE, title=name))
    r = explore_results[name][0]  # seed=0
    heatmap_plots.append(plot_decision_heatmap(r, title=name))

# Context-blind
cb_agent = train_agent_for_viz(W_TRUE, K, n_rounds=n_rounds_explore,
                               hidden_dim=HIDDEN_DIM, loss_type='mse',
                               explore='greedy', full_feedback=False,
                               seed=0)
scatter_plots.append(plot_prediction_scatter(cb_agent, W_TRUE, title='Context-Blind'))
heatmap_plots.append(plot_decision_heatmap(cb_explore[0], title='Context-Blind'))

display(gggrid(scatter_plots, ncol=4) + ggsize(1600, 760))
ggsave(gggrid(scatter_plots, ncol=4) + ggsize(1600, 760), 'figures/sec4_prediction_scatter.svg')
display(gggrid(heatmap_plots, ncol=4) + ggsize(1600, 500))
ggsave(gggrid(heatmap_plots, ncol=4) + ggsize(1600, 500), 'figures/sec4_decision_heatmap.svg')

'/Users/aabhash/Documents/code/DFL for TS Autoregressor/constrained-bandits/notebooks/lets-plot-images/figures/sec4_decision_heatmap.svg'

## 5. The Full System at Scale

Now we understand why each component is needed:
- **Shared linear features**: Enable cross-arm learning (SPO+ needs parameter coupling)
- **Bayesian posterior**: Provides per-arm uncertainty for exploration
- **UCB exploration**: Ensures contested arms are observed
- **Masked SPO+ gradient**: Focuses learning on decision-relevant, observed arms

Scale to **16 arms, select 4** (k/n = 25%) with nonlinear ground truth. Same 2x2 comparison: {MSE, SPO+} x {Greedy, UCB}, with context-blind baseline and full diagnostic panels (regret, flip set, contested/non-contested MAE, decision accuracy, rank correlation, contested uncertainty).


In [9]:
n_arms_big = 16
context_dim_big = 5
k_big = 4  # k/n = 25%
n_rounds_big = 5000
hidden_dim_big = 16

rng_gt2 = np.random.RandomState(42)
W_true_big = rng_gt2.randn(1, context_dim_big) + rng_gt2.randn(n_arms_big, context_dim_big) * 0.4

scale_configs = [
    # (name, loss, explore, beta, phased)
    ('MSE + Greedy',         'mse', 'greedy', 0.0, False),
    ('MSE + UCB',            'mse', 'ucb',    1.0, False),
    ('SPO+ + Greedy',        'spo', 'greedy', 0.0, False),
    ('SPO+ + UCB',           'spo', 'ucb',    1.0, False),
    ('SPO+ + UCB (phased)',  'spo', 'ucb',    1.0, True),
    ('MSE + UCB (phased)',   'mse', 'ucb',    1.0, True),
]

all_results = {}
for name, loss, explore, beta, phased in scale_configs:
    print(f'Running {name}...')
    seeds = []
    for seed in range(N_SEEDS):
        r = run_experiment(W_true_big, k_big, n_rounds=n_rounds_big,
                           hidden_dim=hidden_dim_big, lr=LR,
                           loss_type=loss, explore=explore, beta=beta,
                           mask_grad=True, full_feedback=False, seed=seed,
                           phased=phased)
        seeds.append(r)
    all_results[name] = seeds

print('All experiments complete')

cb_results_16 = run_context_blind(W_true_big, k_big, n_rounds=n_rounds_big, n_seeds=N_SEEDS, hidden_dim=hidden_dim_big, lr=LR)

agent_colors = {
    'MSE + Greedy': '#C0392B',
    'MSE + UCB': '#F1948A',
    'SPO+ + Greedy': '#1A5276',
    'SPO+ + UCB': '#5DADE2',
    'SPO+ + UCB (phased)': '#27AE60',
    'MSE + UCB (phased)': '#F39C12',
    'Context-Blind': 'gray',
}
agent_linestyles = {
    'MSE + Greedy': 'solid', 'MSE + UCB': 'dashed',
    'SPO+ + Greedy': 'solid', 'SPO+ + UCB': 'dashed',
    'SPO+ + UCB (phased)': 'solid',
    'MSE + UCB (phased)': 'solid',
    'Context-Blind': 'dotted',
}

window = 100
kernel_sm = np.ones(window) / window
def smooth(arr):
    return np.convolve(arr, kernel_sm, mode='valid')

def make_scale_plot(metric_key, ylabel, title, include_cb=True):
    dfs = []
    for name, _, _, _, _ in scale_configs:
        vals = np.array([r[metric_key] for r in all_results[name]])
        sv = smooth(vals.mean(axis=0))
        dfs.append(pd.DataFrame({
            'round': np.arange(len(sv)), 'value': sv, 'agent': name,
        }))
    if include_cb:
        cb_vals = np.array([r[metric_key] for r in cb_results_16])
        cb_sv = smooth(cb_vals.mean(axis=0))
        dfs.append(pd.DataFrame({
            'round': np.arange(len(cb_sv)), 'value': cb_sv, 'agent': 'Context-Blind',
        }))
    df = pd.concat(dfs, ignore_index=True)
    return (ggplot(df)
        + geom_line(aes('round', 'value', color='agent', linetype='agent'), size=1.0)
        + scale_color_manual(values=agent_colors)
        + scale_linetype_manual(values=agent_linestyles)
        + labs(x='Round', y=ylabel, title=title)
        + theme(legend_position='bottom', legend_title=element_blank())
        + ggsize(350, 280)
    )

def make_regret_plot_scale(title, include_cb=True):
    dfs = []
    for name, _, _, _, _ in scale_configs:
        regs = np.array([r['regrets'] for r in all_results[name]])
        mean_v = regs.mean(axis=0)
        std_v = regs.std(axis=0)
        dfs.append(pd.DataFrame({
            'round': np.arange(n_rounds_big), 'value': mean_v,
            'lower': mean_v - std_v, 'upper': mean_v + std_v, 'agent': name,
        }))
    if include_cb:
        cb_regs = np.array([r['regrets'] for r in cb_results_16])
        dfs.append(pd.DataFrame({
            'round': np.arange(n_rounds_big), 'value': cb_regs.mean(axis=0),
            'lower': cb_regs.mean(axis=0) - cb_regs.std(axis=0),
            'upper': cb_regs.mean(axis=0) + cb_regs.std(axis=0),
            'agent': 'Context-Blind',
        }))
    df = pd.concat(dfs, ignore_index=True)
    return (ggplot(df)
        + geom_line(aes('round', 'value', color='agent', linetype='agent'), size=1.0)
        + geom_ribbon(aes('round', ymin='lower', ymax='upper', fill='agent'), alpha=0.1, color='rgba(0,0,0,0)')
        + scale_color_manual(values=agent_colors) + scale_fill_manual(values=agent_colors)
        + scale_linetype_manual(values=agent_linestyles)
        + labs(x='Round', y='Cumulative Regret', title=title)
        + theme(legend_position='bottom', legend_title=element_blank())
        + ggsize(350, 280)
    )

def make_perround_regret_scale(title, include_cb=True):
    dfs = []
    for name, _, _, _, _ in scale_configs:
        regs = np.array([r['regrets'] for r in all_results[name]])
        per_round = np.diff(regs, axis=1, prepend=0)
        sv = smooth(per_round.mean(axis=0))
        dfs.append(pd.DataFrame({
            'round': np.arange(len(sv)), 'value': sv, 'agent': name,
        }))
    if include_cb:
        cb_regs = np.array([r['regrets'] for r in cb_results_16])
        cb_per = np.diff(cb_regs, axis=1, prepend=0)
        cb_sv = smooth(cb_per.mean(axis=0))
        dfs.append(pd.DataFrame({
            'round': np.arange(len(cb_sv)), 'value': cb_sv, 'agent': 'Context-Blind',
        }))
    df = pd.concat(dfs, ignore_index=True)
    return (ggplot(df)
        + geom_line(aes('round', 'value', color='agent', linetype='agent'), size=1.0)
        + scale_color_manual(values=agent_colors)
        + scale_linetype_manual(values=agent_linestyles)
        + labs(x='Round', y='Per-Round Regret', title=title)
        + theme(legend_position='bottom', legend_title=element_blank())
        + ggsize(350, 280)
    )

p_a = make_regret_plot_scale('(a) Regret (with baseline)', include_cb=True)
p_b = make_regret_plot_scale('(b) Regret (zoomed)', include_cb=False)
p_c = make_perround_regret_scale('(c) Per-Round Regret', include_cb=True)
p_d = make_scale_plot('contested_mae', 'MAE', '(d) Contested-Arm MAE')
p_e = make_scale_plot('noncontested_mae', 'MAE', '(e) Non-Contested-Arm MAE')
p_f = make_scale_plot('flip_sizes', 'Flip Set Size', '(f) Flip Set Size |F_t|')
p_g = make_scale_plot('decision_correct', 'Decision Accuracy', '(g) Decision Accuracy')
p_h = make_scale_plot('rank_corr', 'Spearman rho', '(h) Rank Correlation')
p_i = make_scale_plot('contested_sigma', 'Mean sigma', '(i) Uncertainty on Contested Arms')

display(gggrid([p_a, p_b, p_c, p_d, p_e, p_f, p_g, p_h, p_i], ncol=3) + ggsize(1200, 900))
ggsave(gggrid([p_a, p_b, p_c, p_d, p_e, p_f, p_g, p_h, p_i], ncol=3) + ggsize(1200, 900), 'figures/sec5_scale.svg')

last = 500
print(f"\n{'Agent':25s} {'Regret':>10s} {'Cntst MAE':>10s} {'NC MAE':>8s} {'DecAcc':>8s} {'Rank rho':>10s} {'|F_t|':>8s}")
print('-' * 85)
for name, _, _, _, _ in scale_configs:
    reg = np.mean([r['regrets'][-1] for r in all_results[name]])
    cmae = np.mean([r['contested_mae'][-last:].mean() for r in all_results[name]])
    ncmae = np.mean([r['noncontested_mae'][-last:].mean() for r in all_results[name]])
    dacc = np.mean([r['decision_correct'][-last:].mean() for r in all_results[name]])
    rho = np.mean([r['rank_corr'][-last:].mean() for r in all_results[name]])
    flip = np.mean([r['flip_sizes'][-last:].mean() for r in all_results[name]])
    print(f'{name:25s} {reg:10.1f} {cmae:10.4f} {ncmae:8.4f} {dacc:8.3f} {rho:10.3f} {flip:8.2f}')


Running MSE + Greedy...
Running MSE + UCB...
Running SPO+ + Greedy...
Running SPO+ + UCB...
Running SPO+ + UCB (phased)...
Running MSE + UCB (phased)...
All experiments complete



Agent                         Regret  Cntst MAE   NC MAE   DecAcc   Rank rho    |F_t|
-------------------------------------------------------------------------------------
MSE + Greedy                  1200.5     0.0892   0.2010    0.147      0.720     7.93
MSE + UCB                     1324.9     0.0834   0.1866    0.082      0.678     7.83
SPO+ + Greedy                  744.6     0.0721   0.1386    0.211      0.748     7.43
SPO+ + UCB                     721.7     0.0697   0.1355    0.194      0.772     7.23
SPO+ + UCB (phased)            208.2     0.0631   0.0918    0.447      0.918     4.92
MSE + UCB (phased)             248.7     0.0602   0.0660    0.488      0.922     2.96


In [10]:
# Prediction scatter & decision heatmaps for Section 5 (n=16, k=4)
scale_viz_configs = [
    # (name, loss, explore, beta, phased)
    ('MSE + Greedy',         'mse', 'greedy', 0.0, False),
    ('MSE + UCB',            'mse', 'ucb',    1.0, False),
    ('SPO+ + Greedy',        'spo', 'greedy', 0.0, False),
    ('SPO+ + UCB',           'spo', 'ucb',    1.0, False),
    ('SPO+ + UCB (phased)',  'spo', 'ucb',    1.0, True),
    ('MSE + UCB (phased)',   'mse', 'ucb',    1.0, True),
]

scatter_plots = []
heatmap_plots = []
for name, loss, explore, beta, phased in scale_viz_configs:
    agent = train_agent_for_viz(W_true_big, k_big, n_rounds=n_rounds_big,
                                hidden_dim=hidden_dim_big, loss_type=loss,
                                explore=explore, beta=beta, mask_grad=True,
                                full_feedback=False, seed=0, phased=phased)
    scatter_plots.append(plot_prediction_scatter(agent, W_true_big, title=name))
    r = all_results[name][0]  # seed=0
    heatmap_plots.append(plot_decision_heatmap(r, title=name))

# Context-blind
cb_agent = train_agent_for_viz(W_true_big, k_big, n_rounds=n_rounds_big,
                               hidden_dim=hidden_dim_big, loss_type='mse',
                               explore='greedy', full_feedback=False,
                               seed=0)
scatter_plots.append(plot_prediction_scatter(cb_agent, W_true_big, title='Context-Blind'))
heatmap_plots.append(plot_decision_heatmap(cb_results_16[0], title='Context-Blind'))

display(gggrid(scatter_plots, ncol=4) + ggsize(1600, 760))
ggsave(gggrid(scatter_plots, ncol=4) + ggsize(1600, 760), 'figures/sec5_prediction_scatter.svg')
display(gggrid(heatmap_plots, ncol=4) + ggsize(1600, 500))
ggsave(gggrid(heatmap_plots, ncol=4) + ggsize(1600, 500), 'figures/sec5_decision_heatmap.svg')

'/Users/aabhash/Documents/code/DFL for TS Autoregressor/constrained-bandits/notebooks/lets-plot-images/figures/sec5_decision_heatmap.svg'

## 6. Component Dissection

The 2x2 comparison (loss x exploration) from Section 6 isolates each contribution:

| | Greedy | UCB |
|---|--------|-----|
| **MSE** | Baseline | +Exploration |
| **SPO+** | +DFL | +DFL+Exploration |

Computes the marginal contribution of each component (exploration, DFL, interaction) from the final regret values above.


In [11]:
regs = {name: np.mean([r['regrets'][-1] for r in all_results[name]]) for name, _, _, _, _ in scale_configs}

mse_g = regs['MSE + Greedy']
mse_u = regs['MSE + UCB']
spo_g = regs['SPO+ + Greedy']
spo_u = regs['SPO+ + UCB']
spo_u_ph = regs.get('SPO+ + UCB (phased)', None)
mse_u_ph = regs.get('MSE + UCB (phased)', None)

print('=== Component Contribution Analysis ===')
print(f'\nBaseline (MSE+Greedy):       {mse_g:.1f}')
print(f'\n  +Exploration (MSE+UCB):    {mse_u:.1f}  ({(mse_g-mse_u)/mse_g*100:+.1f}%)')
print(f'  +DFL only (SPO++Greedy):   {spo_g:.1f}  ({(mse_g-spo_g)/mse_g*100:+.1f}%)')
print(f'  +Both (SPO++UCB):          {spo_u:.1f}  ({(mse_g-spo_u)/mse_g*100:+.1f}%)')
if spo_u_ph is not None:
    print(f'  +Phased (SPO++UCB phased): {spo_u_ph:.1f}  ({(mse_g-spo_u_ph)/mse_g*100:+.1f}%)')
if mse_u_ph is not None:
    print(f'  +Phased (MSE+UCB phased):  {mse_u_ph:.1f}  ({(mse_g-mse_u_ph)/mse_g*100:+.1f}%)')
print(f'\nExploration value (MSE):  {mse_g-mse_u:.1f} regret reduction')
print(f'DFL value (Greedy):       {mse_g-spo_g:.1f} regret reduction')
print(f'DFL value (with UCB):     {mse_u-spo_u:.1f} regret reduction')
if spo_u_ph is not None:
    print(f'Phasing value (SPO+UCB):  {spo_u-spo_u_ph:.1f} regret reduction')
if mse_u_ph is not None:
    print(f'Phasing value (MSE+UCB):  {mse_u-mse_u_ph:.1f} regret reduction')
print(f'\nInteraction: DFL+UCB together = {mse_g-spo_u:.1f} vs sum of parts = {(mse_g-mse_u)+(mse_g-spo_g):.1f}')
synergy = (mse_g - spo_u) - ((mse_g - mse_u) + (mse_g - spo_g))
print(f'Synergy (super-additive if positive): {synergy:+.1f}')
print(f'\nRecall from Section 6: SPO+ masked + Greedy ≈ MSE + Greedy.')
print('The DFL advantage requires exploration to materialize.')

=== Component Contribution Analysis ===

Baseline (MSE+Greedy):       1200.5

  +Exploration (MSE+UCB):    1324.9  (-10.4%)
  +DFL only (SPO++Greedy):   744.6  (+38.0%)
  +Both (SPO++UCB):          721.7  (+39.9%)
  +Phased (SPO++UCB phased): 208.2  (+82.7%)
  +Phased (MSE+UCB phased):  248.7  (+79.3%)

Exploration value (MSE):  -124.3 regret reduction
DFL value (Greedy):       455.9 regret reduction
DFL value (with UCB):     603.1 regret reduction
Phasing value (SPO+UCB):  513.6 regret reduction
Phasing value (MSE+UCB):  1076.2 regret reduction

Interaction: DFL+UCB together = 478.8 vs sum of parts = 331.6
Synergy (super-additive if positive): +147.2

Recall from Section 6: SPO+ masked + Greedy ≈ MSE + Greedy.
The DFL advantage requires exploration to materialize.


## 7. What Controls SPO+ Advantage?

Three factors determine how much SPO+ helps:

1. **Decision margin**: Arms with large margins between selected/unselected
   are easy — any reasonable prediction gives the right decision.
   SPO+ advantage concentrates on **tight-margin** (contested) arms.

2. **Flip set collapse rate**: As predictions improve, the flip set shrinks.
   Faster collapse = less room for SPO+ to differentiate from MSE.

3. **k/n ratio**: Fewer selections → more contested arms near the boundary.

First panel: analytical flip set collapse curve showing how flip set probability depends on prediction error magnitude (2-arm case). Second panel: empirical regret across k/n ratios {1, 2, 4} out of 8, comparing MSE vs SPO+ with UCB exploration.


In [12]:
w_true_2 = np.array([1.0, 1.5])

def simulate_flip_set_collapse(w, n_contexts=200, error_levels=None):
    if error_levels is None:
        error_levels = np.logspace(-2, 1, 50)
    rng = np.random.RandomState(42)
    contexts = rng.randn(n_contexts)
    
    flip_rates, flip_rates_near, flip_rates_far = [], [], []
    
    for sigma in error_levels:
        flips, flips_near, flips_far = 0, 0, 0
        n_near, n_far = 0, 0
        for x in contexts:
            true_costs = w * x
            pred = true_costs + rng.randn(2) * sigma
            _, n_flips = compute_flip_set(pred, true_costs, k=1)
            flips += (n_flips > 0)
            if abs(x) < 1:
                flips_near += (n_flips > 0); n_near += 1
            else:
                flips_far += (n_flips > 0); n_far += 1
        flip_rates.append(flips / n_contexts)
        flip_rates_near.append(flips_near / max(n_near, 1))
        flip_rates_far.append(flips_far / max(n_far, 1))
    return error_levels, flip_rates, flip_rates_near, flip_rates_far

error_levels, fr_all, fr_near, fr_far = simulate_flip_set_collapse(w_true_2)

# (a) Flip set collapse
df_collapse = pd.DataFrame({
    'sigma': np.tile(error_levels, 3),
    'fraction': np.concatenate([fr_all, fr_near, fr_far]),
    'region': (['All contexts'] * len(error_levels) +
               ['Near boundary (|x|<1)'] * len(error_levels) +
               ['Far from boundary (|x|>1)'] * len(error_levels)),
})

p_a = (ggplot(df_collapse)
    + geom_line(aes('sigma', 'fraction', color='region', linetype='region'), size=1.5)
    + scale_x_log10()
    + scale_color_manual(values={'All contexts': 'black', 'Near boundary (|x|<1)': '#d62728', 'Far from boundary (|x|>1)': '#1f77b4'})
    + scale_linetype_manual(values={'All contexts': 'solid', 'Near boundary (|x|<1)': 'dashed', 'Far from boundary (|x|>1)': 'dashed'})
    + labs(x='Prediction error (sigma)', y='Fraction with non-empty flip set',
           title='Flip Set Collapse vs Prediction Accuracy')
    + theme(legend_position='bottom')
    + ggsize(550, 400)
)

# (b) k/n ratio — online vs phased SPO+
kn_results = {}
kn_results_phased = {}
for k_test in [1, 2, 4]:
    # Online
    seeds_res = []
    for seed in range(5):
        r = run_experiment(W_TRUE, k_test, n_rounds=3000, hidden_dim=HIDDEN_DIM,
                           lr=LR, loss_type='spo', explore='ucb', beta=1.0,
                           mask_grad=True, full_feedback=False, seed=seed)
        seeds_res.append(r)
    kn_results[k_test] = seeds_res
    # Phased
    seeds_res_ph = []
    for seed in range(5):
        r = run_experiment(W_TRUE, k_test, n_rounds=3000, hidden_dim=HIDDEN_DIM,
                           lr=LR, loss_type='spo', explore='ucb', beta=1.0,
                           mask_grad=True, full_feedback=False, seed=seed,
                           phased=True)
        seeds_res_ph.append(r)
    kn_results_phased[k_test] = seeds_res_ph

print('=== k/n ratio: online vs phased (final regret) ===')
for k_test in [1, 2, 4]:
    online = np.mean([r['regrets'][-1] for r in kn_results[k_test]])
    phased = np.mean([r['regrets'][-1] for r in kn_results_phased[k_test]])
    print(f'  k={k_test} (k/n={k_test/N_ARMS:.0%}): online={online:.0f}  phased={phased:.0f}')

kn_dfs = []
kn_colors = {f'k={1} (k/n={1/N_ARMS:.0%})': '#d62728',
             f'k={2} (k/n={2/N_ARMS:.0%})': 'black',
             f'k={4} (k/n={4/N_ARMS:.0%})': '#1f77b4'}
for k_test in [1, 2, 4]:
    regs = np.array([r['regrets'] for r in kn_results[k_test]])
    label = f'k={k_test} (k/n={k_test/N_ARMS:.0%})'
    kn_dfs.append(pd.DataFrame({
        'round': np.arange(3000), 'regret': regs.mean(axis=0), 'config': label,
    }))
df_kn = pd.concat(kn_dfs, ignore_index=True)

p_b = (ggplot(df_kn)
    + geom_line(aes('round', 'regret', color='config'), size=1.5)
    + scale_color_manual(values=kn_colors)
    + labs(x='Round', y='Cumulative Regret', title=f'k/n Ratio Controls Difficulty (n={N_ARMS})')
    + theme(legend_position='bottom')
    + ggsize(550, 400)
)

display(gggrid([p_a, p_b], ncol=2) + ggsize(1100, 400))
ggsave(gggrid([p_a, p_b], ncol=2) + ggsize(1100, 400), 'figures/sec7_controlling_factors.svg')

print('Near-boundary contexts keep the flip set alive longer -> more room for SPO+ to help.')
print('Tighter k/n ratio (fewer selections) -> more contested arms -> bigger SPO+ advantage.')


=== k/n ratio: online vs phased (final regret) ===
  k=1 (k/n=12%): online=206  phased=70
  k=2 (k/n=25%): online=192  phased=62
  k=4 (k/n=50%): online=141  phased=48


Near-boundary contexts keep the flip set alive longer -> more room for SPO+ to help.
Tighter k/n ratio (fewer selections) -> more contested arms -> bigger SPO+ advantage.


## Summary

**Same architecture throughout** (shared linear features + Bayesian last layer).
Only the feedback model and exploration strategy change between sections.

**The layer-by-layer build-up:**

1. **Full info**: SPO+ clearly beats MSE by focusing gradients on the flip set (decision-relevant arms)
2. **Partial observability**: We only see selected arms' costs → must impute the rest. Naive SPO+ diverges; **masking** (zeroing gradient on unobserved arms) prevents this.
3. **Masking alone isn't enough**: Greedy selection never observes contested arms → masked SPO+ can't compute its key gradients. **Bayesian exploration** (UCB) ensures contested arms are observed.
4. **At scale**: The pattern holds — SPO+ + UCB consistently beats MSE + UCB.
5. **Context distribution matters**: Narrow contexts keep the flip set alive → SPO+ advantage is largest. Wide contexts make decisions easy → flip set collapses fast.

**The causal chain:**
> Partial observability → need masking → masking blocks learning on unobserved arms → need exploration → Bayesian posterior provides uncertainty → UCB explores contested arms → masked SPO+ gets the observations it needs → decision-focused gradients reduce regret
